In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import r2_score

In [7]:
# Load the dataset
bike_df = pd.read_csv('day (2).csv')

In [8]:
#Drop unnecessary data
bike_df.drop(['instant', 'dteday', 'casual', 'registered'], axis=1, inplace=True)

In [12]:
# 2. Map Categorical Variables
bike_df['season'] = bike_df['season'].map({1: 'spring', 2: 'summer', 3: 'fall', 4: 'winter'})
bike_df['weathersit'] = bike_df['weathersit'].map({1: 'clear', 2: 'mist', 3: 'light_snow', 4: 'heavy_rain'})
bike_df['mnth'] = bike_df['mnth'].map({1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'})

In [13]:
# Create Dummy Variables
bike_df = pd.get_dummies(bike_df, drop_first=True)

In [14]:
# Split Data
df_train, df_test = train_test_split(bike_df, train_size=0.7, test_size=0.3, random_state=100)

In [15]:
# 5. Rescaling the Features (MinMax Scaling)
scaler = MinMaxScaler()
num_vars = ['temp', 'atemp', 'hum', 'windspeed', 'cnt']
df_train[num_vars] = scaler.fit_transform(df_train[num_vars])

print("Training Data Shape:", df_train.shape)

Training Data Shape: (510, 9)


In [18]:
# Divide into X_train and y_train
y_train = df_train.pop('cnt')
X_train = df_train

In [19]:
#Automated Feature Selection with RFE
lm = LinearRegression()
lm.fit(X_train, y_train)

LinearRegression()

In [20]:
#Select features
rfe = RFE(lm, n_features_to_select=15)
rfe = rfe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_rfe.py:300: UserWarning: Found n_features_to_select=15 > n_features=8. There will be no feature selection and all features will be kept.
  warnings.warn(


In [21]:
#Create X_train with RFE selected features
col = X_train.columns[rfe.support_]
X_train_rfe = X_train[col]

In [22]:
#Build Model using Statsmodels (for detailed statistics)
X_train_rfe_sm = sm.add_constant(X_train_rfe)
lm_sm = sm.OLS(y_train, X_train_rfe_sm).fit()

# Print Summary
print(lm_sm.summary())

                            OLS Regression Results                            
Dep. Variable:                    cnt   R-squared:                       0.742
Model:                            OLS   Adj. R-squared:                  0.738
Method:                 Least Squares   F-statistic:                     180.3
Date:                Fri, 09 Jan 2026   Prob (F-statistic):          3.75e-142
Time:                        02:08:27   Log-Likelihood:                 384.17
No. Observations:                 510   AIC:                            -750.3
Df Residuals:                     501   BIC:                            -712.2
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.2905      0.034      8.642      0.0

In [23]:
# Scale Test Set (Transform only, do not fit)
df_test[num_vars] = scaler.transform(df_test[num_vars])

#Split Test Set
y_test = df_test.pop('cnt')
X_test = df_test

In [24]:
#Predict
X_test_new = sm.add_constant(X_test[col])
y_pred = lm_sm.predict(X_test_new)

#Evaluate
r2 = r2_score(y_test, y_pred)
print("R-Squared Score on Test Set:", r2)

R-Squared Score on Test Set: 0.7217143636029884
